In [ ]:
# Install dependencies
!pip install -q sentence-transformers scikit-learn umap-learn plotly

import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
import umap
import plotly.express as px

In [ ]:

# Load dataset
df = pd. read_csv('/content/clinical_notes_diagnosis_prediction_5000.csv')

# Quick exploration
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\\nSample notes:")

In [ ]:
df[['Clinical Notes', 'Diagnosis']].head(3)

In [ ]:
def clean_clinical_note(text):
    text = str(text).lower().strip()
    # Keep medical symbols, numbers, and relevant punctuation
    text = re.sub(r'[^a-z0-9\s/.,mgdL%-]', '', text)
    text = re.sub(r'\s+', ' ', text) # remove extra whitespace
    return text

df['clean_note'] = df['Clinical Notes'].apply(clean_clinical_note)
# Drop rows with missing notes or diagnosis
df = df.dropna(subset=['clean_note', 'Diagnosis']).reset_index(drop=True)

print("Cleaned sample note:")
print(df['clean_note'].iloc[0])

In [ ]:

model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for all clinical notes
note_embeddings = model.encode(
    df['clean_note'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=64
)

print("Embedding shape:", note_embeddings.shape)

In [ ]:
def get_diagnosis_suggestion(new_note, top_k=3, similarity_threshold=0.6):
    """
    Input: New raw clinical note
    Output: Top similar past notes + their diagnoses + similarity scores
    """
    query_clean = clean_clinical_note(new_note)
    query_emb = model.encode([query_clean], normalize_embeddings=True)

    # Compute similarity against all historical notes
    sims = cosine_similarity(query_emb, note_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for idx in top_idx:
        score = sims[idx]
        if score >= similarity_threshold:
            results.append({
                'matched_note': df.iloc[idx]['clinical_note'],
                'diagnosis': df.iloc[idx]['diagnosis'],
                'similarity_to_query': round(float(score), 4)
            })

    if not results:
        return {
            'status': 'No sufficiently similar past cases found',
            'best_match': df.iloc[np.argmax(sims)]['clinical_note'],
            'best_score': round(float(sims.max()), 4)
        }

    return {'status': f'Found {len(results)} similar cases', 'matches': results}

In [ ]:
def find_similar_note_pairs(new_note, top_k=5, pair_similarity_threshold=0.8):
    """
    Returns:
    - For each similar pair: score of A vs query, score of B vs query, similarity between A and B
    """
    query_clean = clean_clinical_note(new_note)
    query_emb = model.encode([query_clean], normalize_embeddings=True)
    sims_to_query = cosine_similarity(query_emb, note_embeddings)[0]
    top_idx = np.argsort(sims_to_query)[::-1][:top_k]

    # Compute pairwise similarity between the top-k candidates
    sub_embeddings = note_embeddings[top_idx]
    pairwise_sim = cosine_similarity(sub_embeddings)

    similar_pairs = []
    for i in range(len(top_idx)):
        for j in range(i+1, len(top_idx)):
            pair_score = pairwise_sim[i,j]
            if pair_score >= pair_similarity_threshold:
                idx_a, idx_b = top_idx[i], top_idx[j]
                similar_pairs.append({
                    'note_A': {
                        'diagnosis': df.iloc[idx_a]['diagnosis'],
                        'note_text': df.iloc[idx_a]['clinical_note'],
                        'score_vs_query': round(float(sims_to_query[idx_a]), 4)
                    },
                    'note_B': {
                        'diagnosis': df.iloc[idx_b]['diagnosis'],
                        'note_text': df.iloc[idx_b]['clinical_note'],
                        'score_vs_query': round(float(sims_to_query[idx_b]), 4)
                    },
                    'similarity_between_A_and_B': round(float(pair_score), 4)
                })

    if not similar_pairs:
        return {'status': 'No highly similar pairs found among top matches'}
    return {'status': f'Found {len(similar_pairs)} similar pair(s)', 'pairs': similar_pairs}

In [ ]:
def analyze_clinical_case(new_note, top_k=5):
    print("="*80)
    print("NEW CLINICAL NOTE INPUT:")
    print(new_note)
    print("="*80)

    # Get diagnosis suggestions
    suggestion_output = get_diagnosis_suggestion(new_note, top_k=top_k)
    print("\\n🔹 DIAGNOSIS SUGGESTIONS FROM SIMILAR CASES:")
    print(suggestion_output['status'])
    for m in suggestion_output.get('matches', []):
        print(f" • Score: {m['similarity_to_query']} | Diagnosis: {m['diagnosis']}")

    # Get similar pairs
    pair_output = find_similar_note_pairs(new_note, top_k=top_k)
    print("\\n🔹 SIMILAR PAIRS (Both Scores + Mutual Similarity):")
    print(pair_output['status'])
    for p in pair_output.get('pairs', []):
        print(f" • A (score={p['note_A']['score_vs_query']}, dx={p['note_A']['diagnosis']}) ↔ B (score={p['note_B']['score_vs_query']}, dx={p['note_B']['diagnosis']}) | A↔B sim={p['similarity_between_A_and_B']}")

In [ ]:
def semantic_clinical_search(query, top_k=5):
    query_emb = model.encode([clean_clinical_note(query)], normalize_embeddings=True)
    sims = cosine_similarity(query_emb, note_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for idx in top_idx:
        results.append({
            'Diagnosis': df.iloc[idx]['Diagnosis'],
            'note': df.iloc[idx]['Clinical Notes'],
            'similarity': round(float(sims[idx]), 4)
        })
    return results

# Example: Search for all asthma cases
print(semantic_clinical_search("asthma exacerbation wheezing shortness of breath", top_k=3))

In [ ]:
le = LabelEncoder()
df['diagnosis_enc'] = le.fit_transform(df['Diagnosis'])
num_classes = len(le.classes_)
print("Diagnosis classes:", le.classes_)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(note_embeddings, df['diagnosis_enc'], test_size=0.2, random_state=42, stratify=df['diagnosis_enc'])

# Train kNN classifier with cosine metric
knn_clf = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn_clf.fit(X_train, y_train)
y_pred = knn_clf.predict(X_test)

print("\\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report

# Get the classification report as a dictionary
report = classification_report(y_test, y_pred, target_names=le.classes_, output_dict=True)

# Convert to DataFrame for easier plotting, excluding 'accuracy', 'macro avg', 'weighted avg'
report_df = pd.DataFrame(report).transpose()
report_df = report_df.drop(columns=['support'])

# Drop overall metrics rows if they exist, keeping only class-wise metrics
class_metrics_df = report_df.drop(index=['accuracy', 'macro avg', 'weighted avg'], errors='ignore')

# Ensure all values are numeric for the heatmap
class_metrics_df = class_metrics_df.astype(float)

# Plotting the heatmap
global heatmap_fig # Declare global to make it accessible to next cell
heatmap_fig, ax = plt.subplots(figsize=(12, 10)) # Assign figure to heatmap_fig
sns.heatmap(class_metrics_df, annot=True, cmap='viridis', fmt=".2f", linewidths=.5, cbar_kws={'label': 'Score'}, ax=ax)
plt.title('Classification Report Heatmap (Precision, Recall, F1-Score per Class)')
plt.xlabel('Metrics')
plt.ylabel('Diagnosis Class')
plt.tight_layout()
plt.show() # Display the plot

In [ ]:
reducer = umap.UMAP(metric='cosine', random_state=42)
embedding_2d = reducer.fit_transform(note_embeddings)

df['umap_x'] = embedding_2d[:,0]
df['umap_y'] = embedding_2d[:,1]

fig = px.scatter(df, x='umap_x', y='umap_y', color='Diagnosis', hover_data=['Clinical Notes'],
                 title='Clinical Note Embeddings by Diagnosis')
fig.show()

In [ ]:
def extract_age(note):
    match = re.search(r'(\d+)-year-old', note)
    if match:
        return int(match.group(1))
    return None

def extract_gender(note):
    if 'male' in note.lower():
        return 'Male'
    elif 'female' in note.lower():
        return 'Female'
    return None

df['age'] = df['Clinical Notes'].apply(extract_age)
df['gender'] = df['Clinical Notes'].apply(extract_gender)
display(df.head())

In [ ]:
# Re-run UMAP with 3 components for 3D visualization
reducer_3d = umap.UMAP(n_components=3, metric='cosine', random_state=42)
embedding_3d = reducer_3d.fit_transform(note_embeddings)

df['umap_x_3d'] = embedding_3d[:,0]
df['umap_y_3d'] = embedding_3d[:,1]
df['umap_z_3d'] = embedding_3d[:,2]

fig_3d = px.scatter_3d(df, x='umap_x_3d', y='umap_y_3d', z='umap_z_3d',
                     color='Diagnosis', hover_data=['Clinical Notes', 'Diagnosis'],
                     title='3D Clinical Note Embeddings by Diagnosis (UMAP)',
                     labels={'umap_x_3d': 'UMAP Dimension 1', 'umap_y_3d': 'UMAP Dimension 2', 'umap_z_3d': 'UMAP Dimension 3'})

fig_3d.update_traces(marker=dict(size=3, opacity=0.7))
fig_3d.update_layout(height=700, showlegend=True)

fig_3d.show()

In [ ]:
print("\nGender distribution:")
display(df['gender'].value_counts())

In [ ]:
# Preprocess numeric demographics
scaler = StandardScaler()
df['age_scaled'] = scaler.fit_transform(df[['age']])
df['gender_enc'] = (df['gender'] == 'Male').astype(int) # binary encode gender

X_demo = df[['age_scaled', 'gender_enc']].values
X_demo_train, X_demo_test = train_test_split(X_demo, test_size=0.2, random_state=42) # align with note split

# Build hybrid Keras model
from tensorflow.keras.layers import Input, Dense, Dropout, Concatenate
from tensorflow.keras.models import Model

note_input = Input(shape=(note_embeddings.shape[1],))
t = Dense(64, activation='relu')(note_input)
t = Dropout(0.3)(t)

demo_input = Input(shape=(X_demo.shape[1],))
n = Dense(16, activation='relu')(demo_input)

combined = Concatenate()([t, n])
z = Dense(32, activation='relu')(combined)
output = Dense(num_classes, activation='softmax')(z)

hybrid_model = Model(inputs=[note_input, demo_input], outputs=output)
hybrid_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train
hybrid_model.fit([X_train, X_demo_train], y_train, validation_data=([X_test, X_demo_test], y_test), epochs=15, batch_size=32)

In [ ]:
def predict_new_note(note_text):
    # Re-use preprocessing functions from earlier cells
    clean_note = clean_clinical_note(note_text)
    age = extract_age(note_text)
    gender = extract_gender(note_text)

    # Scale age (handle missing age gracefully)
    if age is not None:
        age_scaled = scaler.transform(np.array([[age]]))[0][0]
    else:
        # Fallback to mean age if not found in note
        age_scaled = scaler.transform(np.array([[df['age'].mean()]]))[0][0] # Assuming df and scaler are in scope

    # Encode gender
    gender_enc = 1 if gender == 'Male' else 0 # Assuming 'Male' is 1, 'Female' is 0

    demo_features = np.array([[age_scaled, gender_enc]])

    # Generate embeddings
    note_embedding = model.encode([clean_note], normalize_embeddings=True)

    # Make prediction
    predictions = hybrid_model.predict([note_embedding, demo_features])
    predicted_class_index = np.argmax(predictions, axis=1)[0]
    predicted_diagnosis = le.inverse_transform([predicted_class_index])[0] # Assuming le is in scope

    return predicted_diagnosis, predictions

# Example usage:
example_note = "A 72-year-old female presents with progressive shortness of breath, bilateral lower extremity edema, and a history of hypertension. ECG shows left ventricular hypertrophy. Suspected congestive heart failure."
predicted_dx, probs = predict_new_note(example_note)

print(f"New Clinical Note: {example_note}")
print(f"Predicted Diagnosis: {predicted_dx}")
print(f"Prediction Probabilities: {probs}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get all diagnosis labels
diagnosis_labels = le.classes_

# Create a DataFrame for easier plotting
prob_df = pd.DataFrame({
    'Diagnosis': diagnosis_labels,
    'Probability': probs[0]  # Assuming probs is a 2D array like [[p1, p2, ..., pN]]
})

# Sort by probability in descending order
prob_df = prob_df.sort_values(by='Probability', ascending=False)

# Plot the distribution
fig = plt.figure(figsize=(12, 8))
sns.barplot(x='Probability', y='Diagnosis', data=prob_df, palette='viridis')
plt.title('Probability Distribution of Predicted Diagnoses')
plt.xlabel('Probability')
plt.ylabel('Diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select top 5 diagnoses for focused visualization
top_n = 5
prob_df_top_n = prob_df.head(top_n)

fig_top_n_bar = plt.figure(figsize=(10, 6))
sns.barplot(x='Probability', y='Diagnosis', data=prob_df_top_n, palette='coolwarm')
plt.title(f'Top {top_n} Predicted Diagnoses by Probability')
plt.xlabel('Probability')
plt.ylabel('Diagnosis')
plt.xlim(0, 1) # Ensure x-axis goes from 0 to 1 for probability
plt.tight_layout()
plt.show()

In [ ]:
import plotly.express as px

# Create a bar chart with color mapping based on probability, showing only top 5
fig_bar_colormap = px.bar(prob_df.head(5),
                         x='Probability',
                         y='Diagnosis',
                         orientation='h',
                         color='Probability', # Color bars by their probability
                         color_continuous_scale=['pink', 'darkred'], # Use a custom color scale from pink (low) to dark red (high)
                         title='Top 5 Predicted Diagnoses (Color Map)',
                         labels={'Probability': 'Probability', 'Diagnosis': 'Diagnosis'})

# Order bars by probability and set custom x-axis ticks
fig_bar_colormap.update_layout(yaxis={'categoryorder':'total ascending'})
fig_bar_colormap.update_xaxes(tickvals=np.arange(0, 1.01, 0.15)) # Set ticks from 0 to 1 with a step of 0.15
fig_bar_colormap.show()